# 01 · Data checks

This notebook loads the six raw ADEM CSVs in full and checks how the data is structured before any cleaning. It does not change the raw files.

One row is one vacancy–skill pair: a vacancy with five skills appears on five rows.

Checks:

1. First look: columns, types and a few rows of each file
2. Total rows and unique vacancies per file and per year
3. Which months are covered, especially for 2026
4. Whether each vacancy has the same `positions`, canton, occupation and sector on all its rows
5. Whether any `vacancy_id` appears in more than one file
6. Exact duplicate rows and real missing-value rates

The results decide the cleaning rules used in `02_clean_data.ipynb`. Loading all files can take a few minutes.


## 0. Load all files

In [ ]:
from pathlib import Path
import pandas as pd
from pandas.api.types import union_categoricals
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Support running from either the project root or notebooks/.
project_dir = Path.cwd()
if not (project_dir / 'data' / 'raw').is_dir():
    project_dir = project_dir.parent

raw_dir = project_dir / 'data' / 'raw'
files = sorted(raw_dir.glob('*.csv'))
if not files:
    raise FileNotFoundError(f'No CSV files found in {raw_dir.resolve()}')

TEXT_COLS = ['skill', 'skill_uri', 'canton', 'occupation_code',
             'occupation_label', 'nace_code', 'nace_label']
DTYPES = {'vacancy_id': 'int64', 'positions': 'int32',
          'month': 'int8', 'year': 'int16'}
DTYPES.update({col: 'category' for col in TEXT_COLS})
ORIGINAL_COLS = ['vacancy_id', 'skill', 'skill_uri', 'positions', 'month', 'year',
                 'canton', 'occupation_code', 'occupation_label', 'nace_code', 'nace_label']

frames = []
for file in files:
    part = pd.read_csv(file, dtype=DTYPES)
    missing_cols = set(ORIGINAL_COLS) - set(part.columns)
    extra_cols = set(part.columns) - set(ORIGINAL_COLS)
    if missing_cols or extra_cols:
        print(f'WARNING {file.name}: missing {missing_cols}, extra {extra_cols}')
    part['source_file'] = file.stem.replace('datasc-skills-vacancies-', '')
    frames.append(part)
    print(f'{file.name}: {len(part):,} rows')

# Give every text column the same categories in all files so they combine cleanly.
for col in TEXT_COLS:
    cats = union_categoricals([f[col] for f in frames]).categories
    for f in frames:
        f[col] = f[col].cat.set_categories(cats)

df = pd.concat(frames, ignore_index=True)
df['source_file'] = df['source_file'].astype('category')
del frames

print(f'\nTotal: {len(df):,} rows, {df.shape[1]} columns')
print(f'Memory used: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

## 1. First look

Columns, types and the first rows of each file, to see what one row contains.

In [ ]:
print('Columns:', ORIGINAL_COLS)
display(df[ORIGINAL_COLS].dtypes.astype(str).rename('type').to_frame())

for name, part in df.groupby('source_file', observed=True):
    print(f'\n{name}: first 3 rows')
    display(part[ORIGINAL_COLS].head(3))

## 2. Rows, vacancies and posts

- **rows**: vacancy–skill pairs
- **vacancies**: unique `vacancy_id`
- **posts**: `positions` counted once per vacancy (first row), not summed over every row

In [ ]:
# One row per vacancy within each file, used to count vacancies and posts.
vac = df.drop_duplicates(['source_file', 'vacancy_id'])

by_file = pd.DataFrame({
    'rows': df.groupby('source_file', observed=True).size(),
    'vacancies': vac.groupby('source_file', observed=True).size(),
    'posts': vac.groupby('source_file', observed=True)['positions'].sum(),
})
by_file['rows_per_vacancy'] = (by_file['rows'] / by_file['vacancies']).round(2)
print('By file:')
display(by_file)

# Per year. A vacancy is counted in each year it appears in.
vac_year = df.drop_duplicates(['source_file', 'vacancy_id', 'year'])
by_year = pd.DataFrame({
    'rows': df.groupby('year').size(),
    'vacancies': vac_year.groupby('year').size(),
    'posts': vac_year.groupby('year')['positions'].sum(),
})
by_year['rows_per_vacancy'] = (by_year['rows'] / by_year['vacancies']).round(2)
print('By year:')
display(by_year)

# Check that each file only contains the years in its name.
print('Years found in each file:')
display(df.groupby('source_file', observed=True)['year'].unique())

## 3. Months covered

Unique vacancies per year and month. Empty cells or very low counts point to missing or incomplete months. Check 2026 closely before comparing it with full years.

In [ ]:
vac_month = df.drop_duplicates(['source_file', 'vacancy_id', 'year', 'month'])
month_table = pd.crosstab(vac_month['year'], vac_month['month'])
display(month_table)

print('Months with no vacancies (within the years present):')
gaps = [(y, m) for y in month_table.index for m in range(1, 13)
        if m not in month_table.columns or month_table.loc[y, m] == 0]
print(gaps if gaps else 'None')

# Values outside 1-12 would indicate bad data.
bad_months = df.loc[~df['month'].between(1, 12), 'month'].unique()
print('Invalid month values:', bad_months if len(bad_months) else 'None')

## 4. Are vacancy details the same on all rows?

For each vacancy (within a file), count how many distinct values each column takes. Missing values count as a distinct value. If a vacancy is consistent, every column should have exactly 1.

In [ ]:
CHECK_COLS = ['positions', 'month', 'year', 'canton', 'occupation_code', 'nace_code']

distinct = (df.groupby(['source_file', 'vacancy_id'], observed=True)[CHECK_COLS]
              .nunique(dropna=False))

inconsistent = (distinct > 1)
summary = pd.DataFrame({
    'vacancies_with_more_than_one_value': inconsistent.sum(),
    'pct_of_vacancies': (inconsistent.mean() * 100).round(3),
})
display(summary)

print('By file (number of inconsistent vacancies per column):')
display(inconsistent.groupby(level='source_file', observed=True).sum())

In [ ]:
# Show a few examples for each column that has inconsistencies.
for col in CHECK_COLS:
    bad_ids = distinct.index[distinct[col] > 1]
    if len(bad_ids) == 0:
        continue
    print(f'\nExamples where {col} differs within a vacancy:')
    sample_keys = bad_ids[:3]
    example = df.set_index(['source_file', 'vacancy_id']).loc[sample_keys]
    display(example[['skill', 'positions', 'month', 'year', 'canton',
                     'occupation_code', 'nace_code']].head(20))

## 5. Does any vacancy_id appear in more than one file?

If an ID appears in several files, it could be the same vacancy running across periods, or two different vacancies sharing an ID.

In [ ]:
files_per_id = vac.groupby('vacancy_id')['source_file'].nunique()
multi_ids = files_per_id[files_per_id > 1]

print(f'Unique vacancy_ids overall: {len(files_per_id):,}')
print(f'IDs found in more than one file: {len(multi_ids):,} '
      f'({len(multi_ids) / len(files_per_id) * 100:.3f}%)')

if len(multi_ids):
    print('\nNumber of files per repeated ID:')
    display(multi_ids.value_counts().sort_index())

    print('Which file pairs share IDs:')
    shared = vac[vac['vacancy_id'].isin(multi_ids.index)]
    presence = pd.crosstab(shared['vacancy_id'], shared['source_file'].astype(str)) > 0
    pairs = presence.dot(presence.columns + ' + ').str.rstrip(' + ')
    display(pairs.value_counts().head(10))

    print('Examples (one row per vacancy per file):')
    example_ids = multi_ids.index[:5]
    display(vac[vac['vacancy_id'].isin(example_ids)]
            .sort_values(['vacancy_id', 'source_file'])
            [['vacancy_id', 'source_file', 'month', 'year', 'positions',
              'canton', 'occupation_code', 'nace_code']])

## 6. Duplicates and missing values

**Exact duplicates**: rows identical in all 11 original columns. **Repeated skill**: the same skill listed more than once for the same vacancy.

In [ ]:
dup_exact = df.duplicated(subset=['source_file'] + ORIGINAL_COLS)
dup_skill = df['skill_uri'].notna() & df.duplicated(subset=['source_file', 'vacancy_id', 'skill_uri'])

dups = pd.DataFrame({
    'rows': df.groupby('source_file', observed=True).size(),
    'exact_duplicate_rows': dup_exact.groupby(df['source_file'], observed=True).sum(),
    'repeated_skill_rows': dup_skill.groupby(df['source_file'], observed=True).sum(),
})
dups['exact_dup_pct'] = (dups['exact_duplicate_rows'] / dups['rows'] * 100).round(3)
display(dups)

if dup_exact.any():
    print('Example of exact duplicate rows:')
    first_dup = df[dup_exact].iloc[0]
    display(df[(df['source_file'] == first_dup['source_file']) &
               (df['vacancy_id'] == first_dup['vacancy_id'])].head(15))

In [ ]:
# Missing values per column, as a percentage of rows, for each file.
missing = (df[ORIGINAL_COLS].isna()
             .groupby(df['source_file'], observed=True).mean()
             .mul(100).round(2))
print('Missing values (% of rows):')
display(missing.T)

# skill and skill_uri should be missing together.
mismatch = (df['skill'].isna() != df['skill_uri'].isna()).sum()
print(f'Rows where skill and skill_uri disagree on missingness: {mismatch:,}')

In [ ]:
# Vacancies with no skill at all vs. vacancies with some missing skill rows.
skill_status = (df.assign(has_skill=df['skill'].notna())
                  .groupby(['source_file', 'vacancy_id'], observed=True)['has_skill']
                  .agg(['any', 'all']))
skill_summary = pd.DataFrame({
    'vacancies': skill_status.groupby(level='source_file', observed=True).size(),
    'no_skill_at_all': (~skill_status['any']).groupby(level='source_file', observed=True).sum(),
    'some_rows_missing_skill': (skill_status['any'] & ~skill_status['all'])
                                 .groupby(level='source_file', observed=True).sum(),
})
skill_summary['no_skill_pct'] = (skill_summary['no_skill_at_all'] /
                                 skill_summary['vacancies'] * 100).round(2)
display(skill_summary)

In [ ]:
# Codes and labels should match one-to-one.
for code_col, label_col in [('occupation_code', 'occupation_label'), ('nace_code', 'nace_label')]:
    pairs = df[[code_col, label_col]].dropna().drop_duplicates()
    codes_multi = pairs.groupby(code_col, observed=True)[label_col].nunique()
    labels_multi = pairs.groupby(label_col, observed=True)[code_col].nunique()
    print(f'{code_col}: {pairs[code_col].nunique()} codes, '
          f'{(codes_multi > 1).sum()} with more than one label, '
          f'{(labels_multi > 1).sum()} labels with more than one code')

print('\nCanton values and row counts:')
display(df['canton'].value_counts(dropna=False))

## What to note down

After running, write down:

- Rows, vacancies and posts per year, and which months are missing (especially 2026)
- Whether any column differs within a vacancy, and how often
- How many IDs appear in more than one file, and whether those look like the same vacancy
- The share of exact duplicates and of vacancies with no skill
- Which columns have the most missing values, and in which years

Do not delete or fill anything yet. These results will decide the cleaning rules.